### ROUGH DRAFT NOTEBOOK FOR FORECASTING MODEL

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)

In [2]:
path = Path('sq')
projects = {}

for file in sorted(path.glob('*.csv'), key=lambda p: p.name):
    current_df = pd.read_csv(file)
    file_name = file.stem
    projects[file_name] = current_df

In [3]:
# remove blank rows/columns (keep all rows; drop first column only)
for name, df in projects.items():
    df_cleaned = df.iloc[:, 1:].copy()

    # Last row is the Total row; some cells contain a second value on a new line
    if len(df_cleaned) > 0:
        # rename first column value in the last row
        df_cleaned.iat[-1, 0] = "Total"

        # Second value starts appearing at the 7th column (1-indexed)
        start_col_idx = 6  # 0-based index
        if df_cleaned.shape[1] > start_col_idx:
            last_row = df_cleaned.iloc[[-1], start_col_idx:]
            cleaned_last_row = (
                last_row.astype("string")
                .apply(lambda s: s.str.split(r"\r\n|\n|\r", regex=True).str[0])
            )
            df_cleaned.iloc[-1, start_col_idx:] = cleaned_last_row.iloc[0].to_numpy()

    df_cleaned = df_cleaned.reset_index(drop=True)
    projects[name] = df_cleaned

In [4]:
timeline_dates = pd.date_range(start='2011-01-01', end='2027-06-01', freq='MS')
master_timeline = timeline_dates.strftime('%b %Y').tolist()

for name, df in projects.items():
    static_columns = list(df.columns[:7])
    all_desired_columns = static_columns + master_timeline
    df_unified = df.reindex(columns=all_desired_columns, fill_value=0)
    projects[name] = df_unified

In [5]:
# Keep only the "Total" row and drop label columns
for name, df in projects.items():

    # Keep only rows where first column value is "Total"
    total_only = df[df.iloc[:, 0].astype(str).eq("Total")].copy()

    # Drop label columns if they exist
    total_only = total_only.drop(columns=["Line Item", "Description"])

    projects[name] = total_only.reset_index(drop=True)

In [6]:
# Add project code as first column
for name, df in projects.items():
    if df.empty:
        continue

    # Ensure Project Code is the first column
    if "Project Code" in df.columns:
        df = df.drop(columns=["Project Code"])
    df.insert(0, "Project Code", name)

    projects[name] = df

In [7]:
# Combine all projects into one DataFrame
non_empty_projects = [df for df in projects.values() if not df.empty]
all_projects_df = pd.concat(non_empty_projects, ignore_index=True)

In [8]:
# Load gross square footage and merge into all_projects_df
import os

# Adjust path/filename if your CSV lives elsewhere
gross_sq_path = os.path.join(path, "gross_sq.csv") if os.path.exists(os.path.join(path, "gross_sq.csv")) else "gross_sq.csv"

gross_sq = pd.read_csv(gross_sq_path)

# Merge on Project Code
all_projects_df = all_projects_df.merge(
    gross_sq[["Project Code", "Gross Sq Footage"]],
    on="Project Code",
    how="left",
)

# Move Gross Sq Footage to be right after Project Code
if "Gross Sq Footage" in all_projects_df.columns:
    cols = list(all_projects_df.columns)
    cols.insert(1, cols.pop(cols.index("Gross Sq Footage")))
    all_projects_df = all_projects_df[cols]

# Remove Actuals columns
cols_to_drop = ["Actuals To Date", "Actuals + Projections"]
all_projects_df = all_projects_df.drop(columns=cols_to_drop, errors="ignore")

all_projects_df.head(20)

,Project Code,Gross Sq Footage,Projected Budget,Projected Commitments,Estimate at Completion,Jan 2011,Feb 2011,Mar 2011,Apr 2011,May 2011,Jun 2011,Jul 2011,Aug 2011,Sep 2011,Oct 2011,Nov 2011,Dec 2011,Jan 2012,Feb 2012,Mar 2012,Apr 2012,May 2012,Jun 2012,Jul 2012,Aug 2012,Sep 2012,Oct 2012,Nov 2012,Dec 2012,Jan 2013,Feb 2013,Mar 2013,Apr 2013,May 2013,Jun 2013,Jul 2013,Aug 2013,Sep 2013,Oct 2013,Nov 2013,Dec 2013,Jan 2014,Feb 2014,Mar 2014,Apr 2014,May 2014,Jun 2014,Jul 2014,Aug 2014,Sep 2014,Oct 2014,Nov 2014,Dec 2014,Jan 2015,Feb 2015,Mar 2015,Apr 2015,May 2015,Jun 2015,Jul 2015,Aug 2015,Sep 2015,Oct 2015,Nov 2015,Dec 2015,Jan 2016,Feb 2016,Mar 2016,Apr 2016,May 2016,Jun 2016,Jul 2016,Aug 2016,Sep 2016,Oct 2016,Nov 2016,Dec 2016,Jan 2017,Feb 2017,Mar 2017,Apr 2017,May 2017,Jun 2017,Jul 2017,Aug 2017,Sep 2017,Oct 2017,Nov 2017,Dec 2017,Jan 2018,Feb 2018,Mar 2018,Apr 2018,May 2018,Jun 2018,Jul 2018,Aug 2018,Sep 2018,Oct 2018,Nov 2018,Dec 2018,Jan 2019,Feb 2019,Mar 2019,Apr 2019,May 2019,Jun 2019,Jul 2019,Aug 2019,Sep 2019,Oct 2019,Nov 2019,Dec 2019,Jan 2020,Feb 2020,Mar 2020,Apr 2020,May 2020,Jun 2020,Jul 2020,Aug 2020,Sep 2020,Oct 2020,Nov 2020,Dec 2020,Jan 2021,Feb 2021,Mar 2021,Apr 2021,May 2021,Jun 2021,Jul 2021,Aug 2021,Sep 2021,Oct 2021,Nov 2021,Dec 2021,Jan 2022,Feb 2022,Mar 2022,Apr 2022,May 2022,Jun 2022,Jul 2022,Aug 2022,Sep 2022,Oct 2022,Nov 2022,Dec 2022,Jan 2023,Feb 2023,Mar 2023,Apr 2023,May 2023,Jun 2023,Jul 2023,Aug 2023,Sep 2023,Oct 2023,Nov 2023,Dec 2023,Jan 2024,Feb 2024,Mar 2024,Apr 2024,May 2024,Jun 2024,Jul 2024,Aug 2024,Sep 2024,Oct 2024,Nov 2024,Dec 2024,Jan 2025,Feb 2025,Mar 2025,Apr 2025,May 2025,Jun 2025,Jul 2025,Aug 2025,Sep 2025,Oct 2025,Nov 2025,Dec 2025,Jan 2026,Feb 2026,Mar 2026,Apr 2026,May 2026,Jun 2026,Jul 2026,Aug 2026,Sep 2026,Oct 2026,Nov 2026,Dec 2026,Jan 2027,Feb 2027,Mar 2027,Apr 2027,May 2027,Jun 2027
0,5018,99812,"150,000.00","83,533.24","173,866.00",0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.00,0.00,0.00,0.00,0.00,0.00,27.50,440.00,192.50,583.50,758.24,"23,652.00","4,317.50",27.50,0.00,0.00,118.00,0.00,0.00,0.00,0.00,0.00,0.00,965.50,58.00,"6,165.00",0.00,147.00,0.00,0.00,"3,600.00","15,660.00",30.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,125.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,5058,1441000,"509,450,000.00","505,792,437.34","509,574,503.67",0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,"25,546.62","37,569.97","393,450.25","161,115.55","168,243.72","20,321.57","97,341.00","309,912.62","126,153.83","111,020.35","181,750.50","191,789.43","95,238.00","3,933,057.60","3,037,912.11","134,106.83","5,691,053.14","4,745,844.55","7,853,884.62","2,667,278.85","2,673,812.49","4,857,573.82","(3,610,791.46)","2,192,525.64","3,275,556.38","1,740,794.72","7,918,450.18","250,974.64","12,896,147.64","407,285.55","7,673,517.32","8,887,617.94","10,734,725.25","19,709,191.51","12,742,612.59","14,903,007.71","21,232,315.19","25,277,495.64","23,437,879.75","21,435,614.53","388,259.30","51,005,750.24","30,450,346.08","21,283,912.65","36,187,155.46","469,750.24","17,894,873.14","14,087,377.32","9,666,521.51","5,687,009.80","4,012,526.65","3,245,480.45","443,375.51","10,675,546.70","2,797,160.86","9,413,483.33","3,692,330.77","301,540.95","1,051,046.65","82,218.35","111,695.66","31,721.92","806,288.34","33,593.62","84,102.98","102,383.17","3,130,595.11",0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,

# IMMEDIATE NEXT TASKS GANG:

**This is most important bottleneck right now - once we have the database consolidated, the rest of the steps will be quick**

1) compile more project csv's in the "projections" folder.
   - MUST have the entire timeline attached, from project start date to completion date
   - MUST be named as just the project code (ex: C4071.csv)
   - MUST be csv

2) squash categories into only total cost + clean data up all around

3) convert timeline into S-curve parameterization vector (this will be complicated but I will figure it out)

4) squish all projects (which should now have 1 row & 6 columns) into a single dataframe with all 100-300 projects (hopeully on the high-end of this range)

5) we now have a training set!! 💯
